In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

In [2]:
def SOC(I, t, z0, Q_Ah):
    Q = Q_Ah*3600

    delta_t = t.diff().fillna(0)
    dz = I / Q * delta_t
    z = dz.cumsum() + z0
    return z

In [3]:
def soc_from_ocv(U, ocv_file):
    ocv_df = pd.read_csv(ocv_file)

    ocv_df = ocv_df.sort_values("OCV, V")

    U_vals = ocv_df["OCV, V"].values
    SOC_vals = ocv_df["SOC, %"].values 

    # ограничение по диапазону
    U = np.clip(U, U_vals.min(), U_vals.max())

    return np.interp(U, U_vals, SOC_vals)

In [4]:
current_dir = Path.cwd()
print(f"Current dir: {current_dir}")

project_root = current_dir.parent.parent 
low_current_charge_and_discharge_dir = project_root / "Data_preprocessing" / "low_current_charge_and_discharge"
T = ["+25", "+30", "+35"]
start_num = 1
end_num = 6 

Current dir: D:\Documents\battery\Parametrization\USBEREIT conference\Data_processing\Parameters\code


In [5]:
static_parameters = []
data_dir = project_root / "Data_preprocessing" / "low_current_charge_and_discharge"
ocv_out_dir = current_dir.parent / "static_parameters"
ocv_out_dir.mkdir(parents=True, exist_ok=True)


for temp in T:
    for i in range(start_num, end_num):
        # ---------- CHARGE ----------
        df_chg = pd.read_csv(data_dir / f"{i}_{temp}_charge.csv")

        dt = df_chg["t,s"].diff().fillna(0)
        Q_chg = np.sum(df_chg["I,A"] * dt)
        Q_chg_Ah = Q_chg / 3600

        # ---------- DISCHARGE ----------
        df_dchg = pd.read_csv(data_dir / f"{i}_{temp}_discharge.csv")

        dt = df_dchg["t,s"].diff().fillna(0)
        Q_dchg = -np.sum(df_dchg["I,A"] * dt)
        Q_dchg_Ah = Q_dchg / 3600

        # ---------- Coulombic efficiency ----------
        eta = min(Q_dchg / Q_chg, 1.0)

        # ---------- SOC calculation (discharge) ----------
        df_dchg["SOC"] = SOC(
            I=df_dchg["I,A"],
            t=df_dchg["t,s"],
            z0=1.0,               # старт разряда = 100%
            Q_Ah=Q_dchg_Ah
        )

        soc = df_dchg["SOC"]

        soc_start = soc.iloc[0]
        soc_end = soc.iloc[-1]
        
        df_dchg["SOC_norm"] = (soc - soc_end) / (soc_start - soc_end)
        df_dchg["SOC_norm"] = df_dchg["SOC_norm"].clip(0.0, 1.0)
        
        df_dchg["SOC_%"] = df_dchg["SOC_norm"] * 100


        # ---------- OCV(SOC) ----------
        ocv_soc_df = df_dchg[["U,V", "SOC_%"]].rename(
            columns={
                "U,V": "OCV, V",
                "SOC_%": "SOC, %"
            }
        )

        # На всякий случай сортируем по SOC
        ocv_soc_df = ocv_soc_df.sort_values("SOC, %")

        # ---------- Save OCV curve ----------
        OCV_file_name = f"OCV_{i}_{temp}.csv"
        ocv_soc_df.to_csv(ocv_out_dir / OCV_file_name, index=False)

        # ---------- Static parameters ----------
        static_parameters.append({
            "bat_num": i,
            "temp": temp,
            "Q_Ah": Q_dchg_Ah,
            "eta": eta,
            "OCV_file_name": OCV_file_name,
        })

# ---------- Save static parameters ----------
static_parameters_df = pd.DataFrame(static_parameters)
static_parameters_df.to_csv(
    current_dir.parent / "static_parameters" / "static_parameters.csv",
    index=False
)
        
        

In [6]:
profiles_df = pd.read_csv(project_root / "Data_preprocessing" / "profiles" / "profiles_description.csv")
static_df = pd.read_csv( current_dir.parent / "static_parameters" / "static_parameters.csv")


In [7]:
df = profiles_df.merge(
    static_df,
    on=["bat_num", "temp"],
    how="left"
)


In [8]:
SOH_nominal = 3.5  # Ah

SOC_start_list = []
SOC_end_list = []
SOH_list = []

for _, row in df.iterrows():
    ocv_path = ocv_out_dir / row["OCV_file_name"]

    SOC_start = soc_from_ocv(row["U_start_V"], ocv_path)
    SOC_end   = soc_from_ocv(row["U_end_V"], ocv_path)

    SOC_start_list.append(SOC_start)
    SOC_end_list.append(SOC_end)
    SOH_list.append(row["Q_Ah"] / SOH_nominal * 100)


In [9]:
df["SOC_start"] = SOC_start_list
df["SOC_end"] = SOC_end_list
df["SOH"] = SOH_list


In [10]:
df

,bat_num,temp,profile,U_start_V,U_end_V,file_name,Q_Ah,eta,OCV_file_name,SOC_start,SOC_end,SOH
0,1,25,impulse_72_s,4.11093,4.10372,1_+25_impulse_72_s_01.csv,3.289069,0.993163,OCV_1_+25.csv,96.579538,95.849146,93.973411
1,1,25,impulse_144_s,4.10372,4.09365,1_+25_impulse_144_s_02.csv,3.289069,0.993163,OCV_1_+25.csv,95.849146,94.516093,93.973411
2,1,25,impulse_288_s,4.09365,4.07940,1_+25_impulse_288_s_03.csv,3.289069,0.993163,OCV_1_+25.csv,94.516093,91.610379,93.973411
3,1,25,impulse_72_s,4.07940,4.07487,1_+25_impulse_72_s_04.csv,3.289069,0.993163,OCV_1_+25.csv,91.610379,90.357663,93.973411
4,1,25,impulse_144_s,4.07487,4.06139,1_+25_impulse_144_s_05.csv,3.289069,0.993163,OCV_1_+25.csv,90.357663,86.572312,93.973411
...,...,...,...,...,...,...,...,...,...,...,...,...
535,5,35,NEDC,3.95779,3.75069,5_+35_NEDC_02.csv,3.434987,0.991690,OCV_5_+35.csv,76.178929,55.833428,98.142481
536,5,35,NEDC,3.75069,3.59529,5_+35_NEDC_03.csv,3.434987,0.991690,OCV_5_+35.csv,55.833428,36.393882,98.142481
537,5,35,WLTC,4.11297,3.90308,5_+35_WLTC_01.csv,3.434987,0.991690,OCV_5_+35.csv,96.318219,70.142435,98.142481
538,5,35,WLTC,3.90308,3.68861,5_+35_WLTC_02.csv,3.434987,0.991690,OCV_5_+35.csv,70.142435,49.435377,98.142481


In [11]:
df.to_csv(
    current_dir.parent / "static_parameters" / "profiles_description_static_parameters.csv",
    index=False
)